# Flow matching for simple unsupervised distributions

In this notebook we will show how flow matching and rectified flow can be used to sample data from a toy 2D distribution.

In [ ]:
%load_ext autoreload
%autoreload 2

import data  # Auxiliary module for generating toy datasets
import distances  # Auxiliary module for computing distances between distributions
import models  # Auxiliary module for neural network architecture and training
import plotting  # Auxiliary module for visualizations

## Prepare toy data

Generate target distribution data and original noisy data (a standard gaussian)

In [ ]:
import numpy as np

toy_dataset = "two_gaussians"
target_data = data.generate_toy_data(toy_dataset, n=1000)

num_couplings = 100000
source_data = np.random.randn(num_couplings, 2)

plotting.plot_distributions(source_data, target_data)

We can measure the distance between these two distributions quantitatively using the [Wasserstein distance](https://en.wikipedia.org/wiki/Wasserstein_metric), which for empirical distributions involves solving an optimal transportation problem. This has a cubic cost, which can be expensive, but can be afforded for a small sample of data.

In [ ]:
distances.wasserstein_distance(target_data, source_data)

## Basic flow matching

Let's sample independent couplings between source and target data points

In [ ]:
couplings = models.sample_independent_couplings(source_data, target_data, num_couplings=num_couplings)

Visualize the generated couplings

In [ ]:
plotting.plot_distributions(source_data, target_data, couplings=couplings)

Train a simple MLP as the velocity field estimation network.

In [ ]:
network = models.train_flow_model(couplings, verbose=True)

The trained MLP has the following architecture

In [ ]:
plotting.plot_network(network, couplings, save_filename="network_architecture")

Visualize the velocity field we have learned

In [ ]:
plotting.plot_velocity_field(network, source_data, target_data)

Now that we have the velocity field, we can generate new samples from the target distribution by starting form random samples from the source distribution and using a integration method such as Euler's method to run the source samples across the flow.

In [ ]:
source_data = np.random.randn(1000, 2)
trajectories = models.compute_trajectories(network, source_data, n_steps=100, batch_size=2048)

In [ ]:
plotting.animate_trajectories(trajectories, target_data=target_data)

We can compare the generated data with the original target data:

In [ ]:
plotting.plot_generated_data_comparison(target_data, trajectories)

We can now compute the distance between the generated and the target distributions

In [ ]:
generated_data = np.array([traj[-1][1] for traj in trajectories])
distances.wasserstein_distance(target_data, generated_data)

## Rectified flows

By using the generated trajectories to learn a new flow, we can improve the "straightness" of the trajectories, allowing for faster generation. To do so, we genearte new couplings using the newly generated source samples and endpoints of the simulated trajectories.

In [ ]:
source_data = np.random.randn(num_couplings, 2)
trajectories = models.compute_trajectories(network, source_data, n_steps=100, batch_size=2048)
synthetic_data = np.array([traj[-1][1] for traj in trajectories])
couplings = [(src, tgt) for src, tgt in zip(source_data, synthetic_data)]
plotting.plot_distributions(source_data, synthetic_data, couplings=couplings)

In [ ]:
rectified_network = models.train_flow_model(couplings, verbose=True)

Let's check now the rectified velocity field

In [ ]:
plotting.plot_velocity_field(rectified_network, source_data, target_data)

An again we can generate new trajectories using new sampled data, with trajectories that should be straigther in the new field

In [ ]:
source_data = np.random.randn(1000, 2)
rectified_trajectories = models.compute_trajectories(rectified_network, source_data, n_steps=100, batch_size=2048)
plotting.animate_trajectories(rectified_trajectories, target_data=target_data)

In [ ]:
print("Wasserstein distance with rectified flows:", distances.wasserstein_distance(target_data, np.array([traj[-1][1] for traj in rectified_trajectories])))
plotting.plot_generated_data_comparison(target_data, rectified_trajectories)

Since the rectified paths are straighter now, we should be able to generate samples with good quality even if we use fewer Euler steps. As it turns out, we can generate good samples with very few steps.

In [ ]:
plotting.plot_euler_steps_vs_wasserstein_distance({"Flow matching": network, "Rectified flow": rectified_network}, source_data, target_data)

In [ ]:
few_step_trajectories = models.compute_trajectories(rectified_network, source_data, n_steps=8, batch_size=2048)
print("Wasserstein distance with 8 Euler steps:", distances.wasserstein_distance(target_data, np.array([traj[-1][1] for traj in few_step_trajectories])))
plotting.plot_generated_data_comparison(target_data, few_step_trajectories)

## Iterating reflows

Now, we can also run the rectified flow operation any number of times to obtain straighter and straighter paths. For instance, starting from the initial random couplings we can run 5 steps of reflow.

In [ ]:
couplings = models.sample_independent_couplings(source_data, target_data, num_couplings=num_couplings)

num_reflow_steps = 5
best_network = rectified_network
best_couplings = couplings
best_distance = np.inf
for i in range(num_reflow_steps):
    print(f"Reflow step {i + 1}/{num_reflow_steps}...")
    couplings, reflow_network = models.reflow(couplings)
    generated_data = np.array([c[1] for c in couplings])
    distance = distances.wasserstein_distance(target_data, generated_data)
    print(f"Wasserstein distance after reflow step {i + 1}: {distance}")
    if distance < best_distance:
        best_distance = distance
        best_network = reflow_network
        best_couplings = couplings

The Wasserstein distances already tell us something is going badly. As the reflow procedure keeps training on generated data, each iteration biases the generation further away from the original target distribution.

Nevertheless, we can take the best network (the one with smallest Wasserstein distance) and use it to generate data.

In [ ]:
source_data = np.random.randn(1000, 2)
reflow_trajectories = models.compute_trajectories(best_network, source_data, n_steps=100, batch_size=2048)
plotting.animate_trajectories(reflow_trajectories, target_data=target_data)

In [ ]:
print("Wasserstein distance:", distances.wasserstein_distance(target_data, np.array([traj[-1][1] for traj in reflow_trajectories])))
plotting.plot_generated_data_comparison(target_data, reflow_trajectories)

## Distillation

Once the reflow operation has been applied a number of times, we can train a distillation network that directly computes the step to perform to jump from a source data sample to a target data sample.

In [ ]:
distilled_network = models.train_flow_model(best_couplings, verbose=True, train_distilled_network=True)

Then we can generate samples from the target distribution even more very efficiently, in one step

In [ ]:
%%time
source_data = np.random.randn(1000, 2)
generated_data = source_data + models.estimate_velocities(distilled_network, source_data)

In [ ]:
print("Wasserstein distance with distilled network:", distances.wasserstein_distance(target_data, generated_data))
plotting.plot_generated_data_comparison(target_data, generated_data)

## Reverse trajectories

Since the flow is reversible, we can compute reversed trajectories to see how data from the target distribution can be transformed into data from the original distribution.

In [ ]:
reversed_trajectories = models.compute_trajectories(best_network, target_data, n_steps=8, batch_size=2048, reverse=True)
plotting.animate_trajectories(reversed_trajectories, target_data=source_data)

This is particularly useful to compute the degree of anomaly of new data points, as usually the original distribution follows a closed equation for its probability distribution. We can test this with a mesh of points, running their reversed trajectories and then assigning to each starting point a "normality" score following the PDF of the gaussian in the trajectory endpoints.

In [ ]:
mesh_trajectories = models.compute_trajectories(best_network, plotting.mesh_from_data(target_data), n_steps=8, batch_size=2048, reverse=True)
plotting.animate_trajectories(mesh_trajectories)

In [ ]:
plotting.plot_density_map(mesh_trajectories, target_data)

We can also learn a reversed distillation network for these same purposes:

In [ ]:
reversed_distilled_network = models.train_flow_model([(dst, src) for src, dst in best_couplings], verbose=True, train_distilled_network=True)
mesh_trajectories = models.compute_trajectories(reversed_distilled_network, plotting.mesh_from_data(target_data), n_steps=1, batch_size=2048)
plotting.plot_density_map(mesh_trajectories, target_data)